# High-Paying Jobs Data Cleaning & Preparation
This notebook provides a professional, step-by-step workflow for cleaning and preparing U.S. high-paying jobs data. It integrates data from the Bureau of Labor Statistics (BLS) and the U.S. Census to enable robust analysis of salary, education, and occupation across states.

**Workflow Overview:**
- Load and clean BLS data (job and wage statistics)
- Load and clean Census data (demographics, education, occupation)
- Merge datasets for unified analysis
- Save cleaned outputs for downstream analytics

*All steps are clearly commented and follow best practices for reproducibility and clarity.*

# 2.  Data Preparation


In [1]:
# Import required libraries for data processing
import pandas as pd

## BLS Data Preparation

In [2]:
# Load BLS data (job and wage statistics) using openpyxl engine for .xlsx support
bls_data = pd.read_excel("./Resources/bls_state_data.xlsx", engine="openpyxl")
display(bls_data.head(3))  # Preview first rows to confirm successful load

,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,19.88,30.09,46.18,22620,29580,41350,62580,96050,NaN,NaN
1,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,47.95,67.22,95.44,50710,73180,99740,139810,198520,NaN,NaN
2,1,Alabama,2,AL,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,79.48,102.01,#,65700,123960,165320,212180,#,NaN,NaN


In [3]:
# Preview last rows to check for trailing issues or anomalies
display(bls_data.tail(3))

,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
37673,78,Virgin Islands,3,VI,0,Cross-industry,cross-industry,1235,53-7062,"Laborers and Freight, Stock, and Material Move...",...,15.53,17.65,19.42,26520,28920,32300,36720,40390,NaN,NaN
37674,78,Virgin Islands,3,VI,0,Cross-industry,cross-industry,1235,53-7064,"Packers and Packagers, Hand",...,11.56,14.33,17.29,22450,22880,24040,29810,35950,NaN,NaN
37675,78,Virgin Islands,3,VI,0,Cross-industry,cross-industry,1235,53-7065,Stockers and Order Fillers,...,13.12,13.76,16.4,22460,23610,27290,28630,34110,NaN,NaN


In [4]:
# Check column names for consistency and expected structure
bls_data.columns

Index(['AREA', 'AREA_TITLE', 'AREA_TYPE', 'PRIM_STATE', 'NAICS', 'NAICS_TITLE',
       'I_GROUP', 'OWN_CODE', 'OCC_CODE', 'OCC_TITLE', 'O_GROUP', 'TOT_EMP',
       'EMP_PRSE', 'JOBS_1000', 'LOC_QUOTIENT', 'PCT_TOTAL', 'PCT_RPT',
       'H_MEAN', 'A_MEAN', 'MEAN_PRSE', 'H_PCT10', 'H_PCT25', 'H_MEDIAN',
       'H_PCT75', 'H_PCT90', 'A_PCT10', 'A_PCT25', 'A_MEDIAN', 'A_PCT75',
       'A_PCT90', 'ANNUAL', 'HOURLY'],
      dtype='object')

In [5]:
# Select relevant columns for analysis
relevant_columns = [
    'AREA_TITLE', 'PRIM_STATE', 'LOC_QUOTIENT', 'OCC_CODE', 'OCC_TITLE',
    'TOT_EMP', 'JOBS_1000', 'H_MEAN', 'A_MEAN'
# ...add or remove columns as needed for your analysis
]
bls_data = bls_data[relevant_columns]
display(bls_data.head())  # Confirm selection

,AREA_TITLE,PRIM_STATE,LOC_QUOTIENT,OCC_CODE,OCC_TITLE,TOT_EMP,JOBS_1000,H_MEAN,A_MEAN
0,Alabama,AL,1,00-0000,All Occupations,2053090,1000,25.67,53400
1,Alabama,AL,0.74,11-0000,Management Occupations,105580,51.424,56.21,116920
2,Alabama,AL,0.25,11-1011,Chief Executives,720,0.348,106.26,221030
3,Alabama,AL,0.73,11-1021,General and Operations Managers,34450,16.781,62.17,129310
4,Alabama,AL,2.6,11-1031,Legislators,1140,0.555,*,33690


In [6]:
# Check data types to ensure correct formats for analysis
bls_data.dtypes

AREA_TITLE      object
PRIM_STATE      object
LOC_QUOTIENT    object
OCC_CODE        object
OCC_TITLE       object
TOT_EMP         object
JOBS_1000       object
H_MEAN          object
A_MEAN          object
dtype: object

In [7]:
# Create a copy for cleaning and convert numeric columns
bls_df = bls_data.copy()
numeric_columns = ['LOC_QUOTIENT', 'TOT_EMP', 'H_MEAN', 'A_MEAN', 'JOBS_1000']
bls_df[numeric_columns] = bls_df[numeric_columns].apply(pd.to_numeric, errors='coerce')
display(bls_df.head())  # Preview cleaned data

,AREA_TITLE,PRIM_STATE,LOC_QUOTIENT,OCC_CODE,OCC_TITLE,TOT_EMP,JOBS_1000,H_MEAN,A_MEAN
0,Alabama,AL,1.00,00-0000,All Occupations,2053090.0,1000.000,25.67,53400.0
1,Alabama,AL,0.74,11-0000,Management Occupations,105580.0,51.424,56.21,116920.0
2,Alabama,AL,0.25,11-1011,Chief Executives,720.0,0.348,106.26,221030.0
3,Alabama,AL,0.73,11-1021,General and Operations Managers,34450.0,16.781,62.17,129310.0
4,Alabama,AL,2.60,11-1031,Legislators,1140.0,0.555,NaN,33690.0


In [8]:
# Recheck data types after cleaning
bls_df.dtypes

AREA_TITLE       object
PRIM_STATE       object
LOC_QUOTIENT    float64
OCC_CODE         object
OCC_TITLE        object
TOT_EMP         float64
JOBS_1000       float64
H_MEAN          float64
A_MEAN          float64
dtype: object

In [9]:
# Standardize string columns: remove spaces and set title case
bls_df['AREA_TITLE'] = bls_df['AREA_TITLE'].str.strip().str.title()
bls_df['OCC_TITLE'] = bls_df['OCC_TITLE'].str.strip().str.title()

In [10]:
# Clean OCC_CODE: remove hyphens and drop invalid codes
bls_df['OCC_CODE'] = bls_df['OCC_CODE'].str.replace('-', '', regex=False)
bls_df = bls_df[bls_df['OCC_CODE'].notna() & (bls_df['OCC_CODE'] != '')]

In [11]:
# (Optional) Check unique values for each column if needed for EDA
unique_values = bls_df.apply(lambda x: x.unique())

In [12]:
# Check and count missing values in BLS data
missing_values = bls_df.isna().sum()
missing_values

AREA_TITLE         0
PRIM_STATE         0
LOC_QUOTIENT     899
OCC_CODE           0
OCC_TITLE          0
TOT_EMP          899
JOBS_1000        899
H_MEAN          3179
A_MEAN           708
dtype: int64

In [13]:
# Drop rows with any missing values for clean analysis
bls_df = bls_df.dropna(how='any')
bls_df.shape  # Check resulting shape

(33559, 9)

In [14]:
# Keep only the 50 US states (exclude territories and DC)
unwanted_states = ['GU', 'PR', 'VI', 'DC']
bls_clean = bls_df[~bls_df['PRIM_STATE'].isin(unwanted_states)]
bls_clean['PRIM_STATE'].nunique()  # Should be 50

50

In [15]:
# Filter for high-paying jobs: annual mean wage >= $100K or hourly mean >= $48.08
bls_clean = bls_clean[(bls_clean['A_MEAN'] >= 100000) | (bls_clean['H_MEAN'] >= 48.08)]
display(bls_clean.tail(5))

,AREA_TITLE,PRIM_STATE,LOC_QUOTIENT,OCC_CODE,OCC_TITLE,TOT_EMP,JOBS_1000,H_MEAN,A_MEAN
36328,Wyoming,WY,0.72,291229,"Physicians, All Other",400.0,1.463,163.24,339540.0
36330,Wyoming,WY,1.27,291249,"Surgeons, All Other",60.0,0.221,191.58,398480.0
36437,Wyoming,WY,0.48,414011,"Sales Representatives, Wholesale And Manufactu...",270.0,0.990,60.70,126250.0
36527,Wyoming,WY,30.92,475043,"Roof Bolters, Mining",110.0,0.400,51.77,107670.0
36590,Wyoming,WY,3.65,518012,Power Distributors And Dispatchers,60.0,0.217,52.37,108940.0


In [16]:
# Check data types after filtering for high-paying jobs
bls_clean.dtypes

AREA_TITLE       object
PRIM_STATE       object
LOC_QUOTIENT    float64
OCC_CODE         object
OCC_TITLE        object
TOT_EMP         float64
JOBS_1000       float64
H_MEAN          float64
A_MEAN          float64
dtype: object

In [17]:
# Summary statistics for numerical columns in cleaned BLS data
bls_clean.describe()

,LOC_QUOTIENT,TOT_EMP,JOBS_1000,H_MEAN,A_MEAN
count,4259.000000,4.259000e+03,4259.000000,4259.000000,4259.000000
mean,1.103677,1.042200e+04,2.761861,71.987614,149733.226109
std,1.110601,5.237435e+04,9.314721,32.439480,67474.253219
min,0.030000,3.000000e+01,0.005000,48.080000,100000.000000
25%,0.640000,2.900000e+02,0.144000,53.030000,110305.000000
50%,0.920000,1.090000e+03,0.534000,59.770000,124330.000000
75%,1.250000,4.560000e+03,1.625000,73.490000,152865.000000
max,30.920000,1.308800e+06,96.707000,279.590000,581560.000000


In [18]:
# Check structure and info of cleaned BLS data
bls_clean.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 4259 entries, 1 to 36590
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   AREA_TITLE    4259 non-null   object 
 1   PRIM_STATE    4259 non-null   object 
 2   LOC_QUOTIENT  4259 non-null   float64
 3   OCC_CODE      4259 non-null   object 
 4   OCC_TITLE     4259 non-null   object 
 5   TOT_EMP       4259 non-null   float64
 6   JOBS_1000     4259 non-null   float64
 7   H_MEAN        4259 non-null   float64
 8   A_MEAN        4259 non-null   float64
dtypes: float64(5), object(4)
memory usage: 266.2+ KB


In [19]:
# Save cleaned BLS data to CSV for downstream analysis
bls_clean.to_csv('./Data/bls_data.csv', index=False)

## Census Data Preparation

In [20]:
#load data
census_data = pd.read_csv("./Resources/census_data.csv",delimiter=',')
# check data
census_data.head()

,YEAR,REGION,STATEICP,SEX,AGE,EDUC,EDUCD,GRADEATT,GRADEATTD,DEGFIELD,...,OCCSOC,IND,INDNAICS,INCTOT,INCWAGE,OCCSCORE,PWSTATE2,US2022C_WAGP,US2022C_OCCP,US2022C_POWSP
0,2022,32,41,1,25,10,101,7,70,54,...,131111,7870,611M1,6313,6313,45,1,5400,710,1
1,2022,32,41,1,25,10,101,7,70,54,...,131111,7870,611M1,6313,6313,45,1,5400,710,1
2,2022,32,41,1,25,10,101,7,70,54,...,131111,7870,611M1,6313,6313,45,1,5400,710,1
3,2022,32,41,2,24,10,101,7,70,64,...,399041,7870,611M1,11691,11691,13,1,10000,4640,1
4,2022,32,41,1,25,10,101,7,70,54,...,131111,7870,611M1,6313,6313,45,1,5400,710,1


In [21]:
#Check the columns names
census_data.columns

Index(['YEAR', 'REGION', 'STATEICP', 'SEX', 'AGE', 'EDUC', 'EDUCD', 'GRADEATT',
       'GRADEATTD', 'DEGFIELD', 'DEGFIELDD', 'OCC', 'OCCSOC', 'IND',
       'INDNAICS', 'INCTOT', 'INCWAGE', 'OCCSCORE', 'PWSTATE2', 'US2022C_WAGP',
       'US2022C_OCCP', 'US2022C_POWSP'],
      dtype='object')

In [22]:
# Filter for individuals earning $100K or more
census_data= census_data[census_data['INCTOT'] >= 100000]

In [23]:
# filtre the data  to only keep the relevant columns
relevant_c = ['STATEICP' ,'DEGFIELDD', 'EDUCD','OCCSOC','INCTOT','SEX', 'AGE']
census_data=census_data[relevant_c]
census_data.head()

,STATEICP,DEGFIELDD,EDUCD,OCCSOC,INCTOT,SEX,AGE
12,41,2417,101,112022,247841,1,47
13,41,4101,114,551010,112230,1,43
14,41,5901,101,1191XX,149640,2,54
15,41,3401,101,113061,104046,1,36
21,41,6004,114,119030,457102,2,54


In [24]:
# Remove any non-numeric characters and ensure all codes are exactly 6 digits by padding with leading zeros
census_data.loc[:, 'OCCSOC'] = census_data['OCCSOC'].str.extract('(\d+)', expand=False).str.zfill(6)

In [25]:
#Replace all occurrences of 1 with "Male" and 2 with "Female" in the SEX column.#
census_data.loc[:,'SEX'] = census_data['SEX'].astype(str).replace({'1': 'Male', '2': 'Female'})

In [26]:
# Display the data Structure 
census_data.head()

,STATEICP,DEGFIELDD,EDUCD,OCCSOC,INCTOT,SEX,AGE
12,41,2417,101,112022,247841,Male,47
13,41,4101,114,551010,112230,Male,43
14,41,5901,101,001191,149640,Female,54
15,41,3401,101,113061,104046,Male,36
21,41,6004,114,119030,457102,Female,54


In [27]:
# Dictionary to map STATEICP codes to state names (50 U.S. states only)
census_df=census_data.copy()
state_map = {
    1: 'Connecticut', 2: 'Maine', 3: 'Massachusetts', 4: 'New Hampshire',
    5: 'Rhode Island', 6: 'Vermont', 11: 'Delaware', 12: 'New Jersey',
    13: 'New York', 14: 'Pennsylvania', 21: 'Illinois', 22: 'Indiana',
    23: 'Michigan', 24: 'Ohio', 25: 'Wisconsin', 31: 'Iowa', 32: 'Kansas',
    33: 'Minnesota', 34: 'Missouri', 35: 'Nebraska', 36: 'North Dakota',
    37: 'South Dakota', 40: 'Virginia', 41: 'Alabama', 42: 'Arkansas',
    43: 'Florida', 44: 'Georgia', 45: 'Louisiana', 46: 'Mississippi',
    47: 'North Carolina', 48: 'South Carolina', 49: 'Texas', 51: 'Kentucky',
    52: 'Maryland', 53: 'Oklahoma', 54: 'Tennessee', 56: 'West Virginia',
    61: 'Arizona', 62: 'Colorado', 63: 'Idaho', 64: 'Montana', 65: 'Nevada',
    66: 'New Mexico', 67: 'Utah', 68: 'Wyoming', 71: 'California',
    72: 'Oregon', 73: 'Washington', 81: 'Alaska', 82: 'Hawaii'
}
# apply the map on the Data
census_df.loc[:,'STATE']=census_df['STATEICP'].map(state_map)


In [28]:
# Education map for the provided codes
education_map = {

    101: "Bachelor's degree", 
    114: "Master's degree", 
    115: "Professional degree", 
    116: "Doctoral degree", 
}

# Apply the mapping again
census_df['EDUCATION_LABEL'] = census_df['EDUCD'].map(education_map)

In [29]:
degree_field_map = {
    1100: "Agriculture", 1101: "Agriculture Production and Management", 1102: "Agricultural Economics", 1103: "Animal Sciences", 1104: "Food Science",
    1105: "Plant Science", 1106: "Soil Science", 1199: "Miscellaneous Agriculture", 
    1301: "Environmental Science", 1302: "Forestry", 1303: "Natural Resources Management", 
    1401: "Architecture", 
    1501: "Area, Ethnic, and Civilization Studies", 
    1901: "Communications", 1902: "Journalism", 1903: "Mass Media", 1904: "Advertising and Public Relations", 
    2001: "Communication Technologies", 
    2100: "Computer Science and IT", 2101: "Computer Programming", 2102: "Computer Science", 2105: "Information Sciences", 
    2106: "Information Security", 2107: "Computer Networking", 
    2201: "Cosmetology and Culinary Arts", 
    2300: "Education", 2301: "Educational Administration", 2303: "School Counseling", 2304: "Elementary Education", 
    2305: "Mathematics Education", 2306: "Physical and Health Education", 2307: "Early Childhood Education", 
    2308: "Science Education", 2309: "Secondary Education", 2310: "Special Education", 2311: "Social Science Education", 
    2312: "Teacher Education", 2313: "Language and Drama Education", 2314: "Art and Music Education", 2399: "Miscellaneous Education", 
    2400: "Engineering", 2401: "Aerospace Engineering", 2402: "Biological Engineering", 2403: "Architectural Engineering", 
    2404: "Biomedical Engineering", 2405: "Chemical Engineering", 2406: "Civil Engineering", 2407: "Computer Engineering", 
    2408: "Electrical Engineering", 2409: "Environmental Engineering", 2410: "Industrial Engineering", 2411: "Materials Engineering", 
    2412: "Mechanical Engineering", 2413: "Metallurgical Engineering", 2414: "Mining Engineering", 2415: "Naval Architecture", 
    2416: "Nuclear Engineering", 2417: "Petroleum Engineering", 2418: "Miscellaneous Engineering", 2419: "Robotics Engineering", 
    2499: "Engineering Technologies", 2500: "Engineering Tech: General", 2501: "Drafting and Design", 2502: "Electrical Engineering Tech", 
    2503: "Industrial Production Tech", 2504: "Mechanical Engineering Tech", 2599: "Miscellaneous Engineering Tech", 
    2601: "Biology", 2602: "Biochemistry", 2603: "Botany", 
    2901: "Mathematics", 3000: "Statistics", 3301: "Social Work", 3302: "Public Administration", 
    3401: "Physical Sciences", 3402: "Astronomy", 3600: "Chemistry", 3601: "Geology", 3602: "Geosciences", 3603: "Oceanography", 
    3604: "Physics", 3605: "Meteorology", 3606: "Planetary Science", 3607: "Materials Science", 3608: "Atmospheric Science", 
    3609: "Environmental Science", 3611: "Space Science", 3699: "Misc. Physical Sciences", 
    3700: "Social Sciences", 3701: "Anthropology", 3702: "Archaeology", 
    3801: "Philosophy and Religion", 
    4000: "Economics", 4001: "Sociology", 4002: "Political Science", 4005: "Geography", 4006: "Criminal Justice", 
    4007: "Public Policy", 4801: "Law and Legal Studies", 
    5000: "Liberal Arts", 5001: "English Literature", 5002: "Linguistics", 5003: "Foreign Languages", 5004: "Comparative Literature", 
    5005: "Philosophy", 5006: "Religious Studies", 5007: "Humanities", 5008: "Interdisciplinary Humanities", 5098: "Miscellaneous Humanities", 
    5102: "Music", 5200: "Fine Arts", 5201: "Design and Applied Arts", 5202: "Graphic Design", 5203: "Photography", 5205: "Film", 5206: "Theater Arts", 
    5299: "Miscellaneous Fine Arts", 
    5401: "Business", 5402: "Accounting", 5403: "Finance", 5404: "Human Resources", 
    5500: "Management Information Systems", 5501: "Marketing", 5502: "Management Science", 5503: "Supply Chain Management", 
    5504: "Real Estate", 5505: "Entrepreneurship", 5506: "International Business", 5507: "Business Administration", 5599: "Miscellaneous Business", 
    5601: "Consumer Sciences", 5701: "Public Health", 
    6000: "Nursing", 6001: "Pharmacy", 6002: "Health Administration", 6003: "Public Health", 6004: "Medical Assisting", 
    6005: "Clinical Sciences", 6006: "Veterinary Sciences", 6007: "Health Services", 6099: "Miscellaneous Health Professions", 
    6100: "General Medicine", 6102: "Dentistry", 6103: "Pharmacy Technician", 6104: "Radiological Sciences", 6199: "Miscellaneous Medical Sciences", 
    6105: "Biomedical Sciences", 6106: "Nutritional Science", 6107: "Athletic Training", 6108: "Therapeutic Sciences", 
    6109: "Mental Health Services", 6110: "Community Health Services", 
    6200: "Psychology", 6201: "Clinical Psychology", 6202: "Industrial Psychology", 6203: "Developmental Psychology", 
    6204: "Cognitive Psychology", 6205: "Forensic Psychology", 6206: "Organizational Psychology", 6207: "Educational Psychology", 
    6209: "Experimental Psychology", 6210: "Social Psychology", 6211: "Counseling Psychology", 6212: "Abnormal Psychology", 6299: "Miscellaneous Psychology", 
    6402: "Human Development", 
    6403: "Social Work", 3202: "Military Science and Leadership", 4901: "Mechanical Engineering", 4101: "General Engineering", 
    5901: "Educational Leadership", 5301: "Business Administration", 3501: "Health Professions"
}
census_df['DEGFIELDD_NAME'] = census_df['DEGFIELDD'].map(degree_field_map)

In [30]:
# check missing  values
census_df.isna().sum()

STATEICP           0
DEGFIELDD          0
EDUCD              0
OCCSOC             0
INCTOT             0
SEX                0
AGE                0
STATE              0
EDUCATION_LABEL    0
DEGFIELDD_NAME     0
dtype: int64

In [31]:
#Drop missing valeus
census_df=census_df.dropna(how='any')

In [32]:
# rename the occupation code
census_df=census_df.rename(columns={'OCCSOC':'OCC_CODE'})

In [33]:
#check data structure
census_df.head()

,STATEICP,DEGFIELDD,EDUCD,OCC_CODE,INCTOT,SEX,AGE,STATE,EDUCATION_LABEL,DEGFIELDD_NAME
12,41,2417,101,112022,247841,Male,47,Alabama,Bachelor's degree,Petroleum Engineering
13,41,4101,114,551010,112230,Male,43,Alabama,Master's degree,General Engineering
14,41,5901,101,001191,149640,Female,54,Alabama,Bachelor's degree,Educational Leadership
15,41,3401,101,113061,104046,Male,36,Alabama,Bachelor's degree,Physical Sciences
21,41,6004,114,119030,457102,Female,54,Alabama,Master's degree,Medical Assisting


In [34]:
#save cleand census data 
census_df.to_csv('./Data/census_data.csv',index=False)

## 3. Data Merging :Combining Census and BLS Data

In [35]:

# Rename columns in the BLS DataFrame for clarity
bls_clean.rename(columns={'AREA_TITLE': 'STATE'}, inplace=True)

In [36]:
# Merge the DataFrames on OCC_CODE and STATE 
merged_df = pd.merge(census_df, bls_clean, on=['OCC_CODE', 'STATE'], how='inner')

In [37]:
# check if is there any missing values before procedding 
merged_df.isna().sum()

STATEICP           0
DEGFIELDD          0
EDUCD              0
OCC_CODE           0
INCTOT             0
SEX                0
AGE                0
STATE              0
EDUCATION_LABEL    0
DEGFIELDD_NAME     0
PRIM_STATE         0
LOC_QUOTIENT       0
OCC_TITLE          0
TOT_EMP            0
JOBS_1000          0
H_MEAN             0
A_MEAN             0
dtype: int64

In [38]:
#check the data Structure 
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 10273 entries, 0 to 10272
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   STATEICP         10273 non-null  int64  
 1   DEGFIELDD        10273 non-null  int64  
 2   EDUCD            10273 non-null  int64  
 3   OCC_CODE         10273 non-null  object 
 4   INCTOT           10273 non-null  int64  
 5   SEX              10273 non-null  object 
 6   AGE              10273 non-null  int64  
 7   STATE            10273 non-null  object 
 8   EDUCATION_LABEL  10273 non-null  object 
 9   DEGFIELDD_NAME   10273 non-null  object 
 10  PRIM_STATE       10273 non-null  object 
 11  LOC_QUOTIENT     10273 non-null  float64
 12  OCC_TITLE        10273 non-null  object 
 13  TOT_EMP          10273 non-null  float64
 14  JOBS_1000        10273 non-null  float64
 15  H_MEAN           10273 non-null  float64
 16  A_MEAN           10273 non-null  float64
dtypes: float64(5

In [39]:
# display the data Statistic
merged_df.describe()

,STATEICP,DEGFIELDD,EDUCD,INCTOT,AGE,LOC_QUOTIENT,TOT_EMP,JOBS_1000,H_MEAN,A_MEAN
count,10273.000000,10273.000000,10273.000000,1.027300e+04,10273.000000,10273.000000,10273.000000,10273.000000,10273.000000,10273.000000
mean,42.615789,4457.860703,107.057724,1.680939e+05,41.309257,1.161845,63303.016646,6.993090,65.769981,136800.167429
std,24.468860,1751.341420,6.657895,1.051136e+05,10.944780,0.539022,91276.508070,7.223279,13.800846,28705.399697
min,1.000000,1100.000000,101.000000,1.000000e+05,19.000000,0.080000,50.000000,0.018000,48.080000,100000.000000
25%,14.000000,2408.000000,101.000000,1.148220e+05,32.000000,0.890000,7890.000000,1.678000,54.940000,114270.000000
50%,45.000000,5205.000000,101.000000,1.367800e+05,40.000000,1.030000,21870.000000,3.757000,62.970000,130970.000000
75%,71.000000,6107.000000,114.000000,1.750000e+05,49.000000,1.340000,79050.000000,10.216000,73.500000,152880.000000
max,82.000000,6403.000000,116.000000,1.098315e+06,94.000000,8.960000,448530.000000,33.810000,123.150000,256160.000000


In [40]:
merged_df.columns

Index(['STATEICP', 'DEGFIELDD', 'EDUCD', 'OCC_CODE', 'INCTOT', 'SEX', 'AGE',
       'STATE', 'EDUCATION_LABEL', 'DEGFIELDD_NAME', 'PRIM_STATE',
       'LOC_QUOTIENT', 'OCC_TITLE', 'TOT_EMP', 'JOBS_1000', 'H_MEAN',
       'A_MEAN'],
      dtype='object')

In [41]:
# Define the desired column order (update to match actual columns in merged_df)

# Get the actual columns present in merged_df

actual_columns = list(merged_df.columns)

# List of columns we want, in preferred order (will only keep those that exist)

desired_order = [
    'PRIM_STATE', 'STATE',   # Geographic Information
    'SEX', 'AGE', 'EDUCD', 'EDUCATION_LABEL', 'DEGFIELDD',  # Demographics and Education
    'OCC_CODE', 'OCC_TITLE',  # Occupation Details
    'INCTOT', 'TOT_EMP', 'LOC_QUOTIENT', 'JOBS_1000',  # Employment/Income Statistics
    'H_MEAN', 'A_MEAN'  # Wage Information
]

# Filter desired_order to only include columns that exist in merged_df

final_order = [col for col in desired_order if col in actual_columns]

if not final_order:

    raise ValueError('No matching columns found for reordering. Please check merged_df columns.')

# Reindex the DataFrame with the filtered order

merged_df = merged_df[final_order]


# Renaming columns for clarity (no more than two words)

rename_mapping = {
    'PRIM_STATE': 'State Abbreviation',
    'STATE': 'State',
    'SEX': 'Gender',
    'AGE': 'Age',
    'EDUCD': 'Education Code',
    'EDUCATION_LABEL': 'Education Level',
    'DEGFIELDD': 'Degree Field',
    'OCC_CODE': 'Occupation Code',
    'OCC_TITLE': 'Occupation',
    'INCTOT': 'Annual Income',
    'TOT_EMP': 'Employment',
    'LOC_QUOTIENT': 'Location Quotient',
    'JOBS_1000': 'Jobs per 1000',
    'H_MEAN': 'Hourly Mean',
    'A_MEAN': 'Annual Mean Wage'
}

# Only rename columns that exist in merged_df

rename_mapping = {k: v for k, v in rename_mapping.items() if k in merged_df.columns}

merged_df.rename(columns=rename_mapping, inplace=True)

# Display the updated DataFrame columns for verification

print('Final columns:', list(merged_df.columns))

Final columns: ['State Abbreviation', 'State', 'Gender', 'Age', 'Education Code', 'Education Level', 'Degree Field', 'Occupation Code', 'Occupation', 'Annual Income', 'Employment', 'Location Quotient', 'Jobs per 1000', 'Hourly Mean', 'Annual Mean Wage']


In [42]:
# Display the updated DataFrame columns as a list for better readability
print(list(merged_df.columns))

['State Abbreviation', 'State', 'Gender', 'Age', 'Education Code', 'Education Level', 'Degree Field', 'Occupation Code', 'Occupation', 'Annual Income', 'Employment', 'Location Quotient', 'Jobs per 1000', 'Hourly Mean', 'Annual Mean Wage']


In [43]:
merged_df.head()

,State Abbreviation,State,Gender,Age,Education Code,Education Level,Degree Field,Occupation Code,Occupation,Annual Income,Employment,Location Quotient,Jobs per 1000,Hourly Mean,Annual Mean Wage
0,AL,Alabama,Male,47,101,Bachelor's degree,2417,112022,Sales Managers,247841,3650.0,0.47,1.778,66.20,137700.0
1,AL,Alabama,Female,31,101,Bachelor's degree,6200,112022,Sales Managers,151978,3650.0,0.47,1.778,66.20,137700.0
2,AL,Alabama,Male,34,101,Bachelor's degree,2599,112022,Sales Managers,149268,3650.0,0.47,1.778,66.20,137700.0
3,AL,Alabama,Male,65,101,Bachelor's degree,5501,112022,Sales Managers,200000,3650.0,0.47,1.778,66.20,137700.0
4,AL,Alabama,Male,36,101,Bachelor's degree,3401,113061,Purchasing Managers,104046,1360.0,1.30,0.665,59.61,123990.0


In [44]:
# Check for duplicate rows in the final dataset before saving
num_duplicates = merged_df.duplicated().sum()
print('Number of duplicate rows before saving:', num_duplicates)
# Remove duplicate rows from the final dataset
merged_df = merged_df.drop_duplicates()
print('Shape after removing duplicates:', merged_df.shape)

Number of duplicate rows before saving: 18
Shape after removing duplicates: (10255, 15)


In [45]:
# Save the cleaned data to a CSV file
merged_df.to_csv('./Data/cleaned_high_pay_data.csv', index=False)

## Final Data Validation and Summary
A quick check to confirm the final cleaned dataset is as expected.

In [46]:
# Show shape and a sample of the cleaned, merged data
print('Final dataset shape:', merged_df.shape)
display(merged_df.head())
# Check for missing values in any column
print('Missing values per column:')
print(merged_df.isna().sum())

Final dataset shape: (10255, 15)


,State Abbreviation,State,Gender,Age,Education Code,Education Level,Degree Field,Occupation Code,Occupation,Annual Income,Employment,Location Quotient,Jobs per 1000,Hourly Mean,Annual Mean Wage
0,AL,Alabama,Male,47,101,Bachelor's degree,2417,112022,Sales Managers,247841,3650.0,0.47,1.778,66.20,137700.0
1,AL,Alabama,Female,31,101,Bachelor's degree,6200,112022,Sales Managers,151978,3650.0,0.47,1.778,66.20,137700.0
2,AL,Alabama,Male,34,101,Bachelor's degree,2599,112022,Sales Managers,149268,3650.0,0.47,1.778,66.20,137700.0
3,AL,Alabama,Male,65,101,Bachelor's degree,5501,112022,Sales Managers,200000,3650.0,0.47,1.778,66.20,137700.0
4,AL,Alabama,Male,36,101,Bachelor's degree,3401,113061,Purchasing Managers,104046,1360.0,1.30,0.665,59.61,123990.0


Missing values per column:
State Abbreviation    0
State                 0
Gender                0
Age                   0
Education Code        0
Education Level       0
Degree Field          0
Occupation Code       0
Occupation            0
Annual Income         0
Employment            0
Location Quotient     0
Jobs per 1000         0
Hourly Mean           0
Annual Mean Wage      0
dtype: int64
